In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

In [0]:
CATALOG = "workspace"
SCHEMA = "final"

BRONZE_TABLE = f"{CATALOG}.{SCHEMA}.bronze_sales_order_header"
SILVER_TABLE = f"{CATALOG}.{SCHEMA}.silver_sales_order_header"

In [0]:
sales_df = spark.table(BRONZE_TABLE)

display(sales_df)

In [0]:
print(
    f"Bronze Count : {sales_df.count()}"
)

In [0]:
def clean_text(column_name):

    cleaned = trim(
        regexp_replace(
            regexp_replace(
                col(column_name),
                "\\+",
                ""
            ),
            "&",
            ""
        )
    )

    return when(
        cleaned == "",
        None
    ).otherwise(cleaned)

In [0]:
null_analysis_df = sales_df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in sales_df.columns
])

display(null_analysis_df)

In [0]:
sales_clean_df = (
    sales_df

    .withColumn(
        "SalesOrderNumber",
        clean_text("SalesOrderNumber")
    )

    .withColumn(
        "PurchaseOrderNumber",
        clean_text("PurchaseOrderNumber")
    )

    .withColumn(
        "AccountNumber",
        clean_text("AccountNumber")
    )

    .withColumn(
        "CreditCardApprovalCode",
        clean_text("CreditCardApprovalCode")
    )

    .withColumn(
        "Comment",
        clean_text("Comment")
    )

    .withColumn(
        "rowguid",
        clean_text("rowguid")
    )
)

In [0]:
sales_clean_df = (
    sales_clean_df

    .withColumn(
        "SalesOrderID",
        expr("try_cast(SalesOrderID as BIGINT)")
    )

    .withColumn(
        "RevisionNumber",
        expr("try_cast(RevisionNumber as INT)")
    )

    .withColumn(
        "Status",
        expr("try_cast(Status as INT)")
    )

    .withColumn(
        "OnlineOrderFlag",
        expr("try_cast(OnlineOrderFlag as INT)")
    )

    .withColumn(
        "CustomerID",
        expr("try_cast(CustomerID as BIGINT)")
    )

    .withColumn(
        "SalesPersonID",
        expr("try_cast(SalesPersonID as BIGINT)")
    )

    .withColumn(
        "TerritoryID",
        expr("try_cast(TerritoryID as INT)")
    )

    .withColumn(
        "BillToAddressID",
        expr("try_cast(BillToAddressID as BIGINT)")
    )

    .withColumn(
        "ShipToAddressID",
        expr("try_cast(ShipToAddressID as BIGINT)")
    )

    .withColumn(
        "ShipMethodID",
        expr("try_cast(ShipMethodID as INT)")
    )

    .withColumn(
        "CreditCardID",
        expr("try_cast(CreditCardID as BIGINT)")
    )

    .withColumn(
        "CurrencyRateID",
        expr("try_cast(CurrencyRateID as BIGINT)")
    )

    .withColumn(
        "SubTotal",
        expr("try_cast(SubTotal as DOUBLE)")
    )

    .withColumn(
        "TaxAmt",
        expr("try_cast(TaxAmt as DOUBLE)")
    )

    .withColumn(
        "Freight",
        expr("try_cast(Freight as DOUBLE)")
    )

    .withColumn(
        "TotalDue",
        expr("try_cast(TotalDue as DOUBLE)")
    )
)

In [0]:
sales_clean_df = (
    sales_clean_df

    .withColumn(
        "OrderDate",
        expr("try_cast(OrderDate as timestamp)")
    )

    .withColumn(
        "DueDate",
        expr("try_cast(DueDate as timestamp)")
    )

    .withColumn(
        "ShipDate",
        expr("try_cast(ShipDate as timestamp)")
    )

    .withColumn(
        "ModifiedDate",
        expr("try_cast(ModifiedDate as timestamp)")
    )
)

In [0]:
sales_valid_df = (
    sales_clean_df

    .filter(
        col("SalesOrderID").isNotNull()
    )

    .filter(
        col("CustomerID").isNotNull()
    )

    .filter(
        col("TotalDue") > 0
    )
)

In [0]:
window_spec = (
    Window
    .partitionBy("SalesOrderID")
    .orderBy(col("ingested_at").desc())
)

sales_dedup_df = (
    sales_valid_df
    .withColumn(
        "rn",
        row_number().over(window_spec)
    )
    .filter(col("rn") == 1)
    .drop("rn")
)

In [0]:
dq_orderid = sales_dedup_df.filter(
    col("SalesOrderID").isNull()
).count()

dq_customer = sales_dedup_df.filter(
    col("CustomerID").isNull()
).count()

dq_totaldue = sales_dedup_df.filter(
    col("TotalDue") <= 0
).count()

dq_dup = (
    sales_dedup_df
    .groupBy("SalesOrderID")
    .count()
    .filter(col("count") > 1)
    .count()
)

print(f"SalesOrderID Nulls      : {dq_orderid}")
print(f"CustomerID Nulls        : {dq_customer}")
print(f"Invalid TotalDue        : {dq_totaldue}")
print(f"Duplicate Sales Orders  : {dq_dup}")

In [0]:
sales_dedup_df = (
    sales_dedup_df
    .withColumn(
        "sales_header_hash",
        sha2(
            concat_ws(
                "|",
                coalesce(col("CustomerID").cast("string"), lit("")),
                coalesce(col("TerritoryID").cast("string"), lit("")),
                coalesce(col("SubTotal").cast("string"), lit("")),
                coalesce(col("TaxAmt").cast("string"), lit("")),
                coalesce(col("Freight").cast("string"), lit("")),
                coalesce(col("TotalDue").cast("string"), lit(""))
            ),
            256
        )
    )
)

In [0]:
silver_sales_df = (
    sales_dedup_df
    .withColumn(
        "silver_load_timestamp",
        current_timestamp()
    )
    .withColumn(
        "silver_layer",
        lit("SILVER")
    )
)

In [0]:
(
    silver_sales_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema","true")
    .saveAsTable(SILVER_TABLE)
)

In [0]:
silver_df = spark.table(SILVER_TABLE)

print(
    f"Silver Count : {silver_df.count()}"
)

display(silver_df)